Proof of <a class="ProveItLink" href="../../../../../../../_theory_nbs_/theory.ipynb">proveit</a>.<a class="ProveItLink" href="../../../../../../_theory_nbs_/theory.ipynb">physics</a>.<a class="ProveItLink" href="../../../../../_theory_nbs_/theory.ipynb">quantum</a>.<a class="ProveItLink" href="../../../../_theory_nbs_/theory.ipynb">QEC</a>.<a class="ProveItLink" href="../../theory.ipynb">kitaev_planar_hadamard</a>.<a class="ProveItLink" href="../../theorems.ipynb#rbb_verts_nonempty">rbb_verts_nonempty</a> theorem
========

In [ ]:
import proveit
theory = proveit.Theory() # the theorem's theory
from proveit import defaults, display_provers # useful imports
from proveit.physics.quantum.QEC.kitaev_planar_hadamard import RoughBoundaryBVertices

In [ ]:
%proving rbb_verts_nonempty

In [ ]:
RoughBoundaryBVertices.definition()

To show that the `UnionAll` is non-empty, we need to show that its “contents” are non-empty for at least one value of $t \in \{0, \ldots, n\}$. Our basic approach here is to show that (1) $\{(0, \ell, t) | \ell \in \{1-\delta_{t,0}, \ldots, n-1-t\}\} \ne \emptyset$ when we use $t = 0$; then (2) show that the extended union $\{(0, \ell, t) | \ell \in \{1-\delta_{t,0}, \ldots, n-1-t\}\} \cup \{(m, 0, t) | m \in \{1 - \delta_{t,(n-1)} - \delta_{t,n}, \ldots, t\}\} \ne \emptyset$ when we use $t=0$; and then conclude that the entire `UnionAll` is non-empty (since its “contents” are non-empty for at least one value in the `UnionAll`'s domain $\{0, \ldots, n\}$).

From inspection, we can see that the first component $\{(0, \ell, t) | \ell \in \{1-\delta_{t,0}, \ldots, n-1-t\}\}$ contains the 3-tuple $(0, 0, 0)$ and is thus non-empty when $t = 0$. We don't want to carry around an _assumption_ that $t = 0$, however (that makes things difficult); instead, we consider the specific subset $\{(0, \ell, 0) | \ell \in \{0, \ldots, n-1\}\}$.

In [ ]:
union_all = RoughBoundaryBVertices.definition().rhs

In [ ]:
RoughBoundaryBVertices.definition().rhs.instance_expr.operands[0]

In [ ]:
from proveit.physics.quantum.QEC.kitaev_planar_hadamard import _n_in_integer

In [ ]:
from proveit import t, ExprTuple
from proveit.logic.sets import SetOfAll
from proveit.numbers import zero, one, Interval, subtract
from proveit.physics.quantum.QEC.kitaev_planar_hadamard import ell, _n
A_zero = SetOfAll(ell, ExprTuple(zero, ell, zero), domain=Interval(zero, subtract(_n, one)))

##### Now let's prove that $(0, 0, 0) \in A_{zero} = \{(n, \ell, 0)\}_{\ell \in \{0, \ldots, n-1\}}$. We do this via an existential in the `InSet` definition.

In [ ]:
selected_tuple = ExprTuple(zero, zero, zero)

In [ ]:
from proveit.logic.sets import InSet
inset_def = InSet(selected_tuple, A_zero).definition()

##### But we need a little more information about $0$ and the domain $\{0, \ldots, n-1\}$.

In [ ]:
from proveit.physics.quantum.QEC.kitaev_planar_hadamard import _n_ge_three, _n_in_integer
display(_n_ge_three)
display(_n_in_integer)

In [ ]:
InSet(zero, inset_def.rhs.domain).prove()

In [ ]:
exists_ell_judgment = inset_def.rhs.conclude_via_example(zero)

In [ ]:
selected_tuple_in_set = inset_def.sub_left_side_into(exists_ell_judgment)

In [ ]:
# this is not yet fully automated, because we still need to update Exists.conclude_via_example()
# with a new, simpler theorem;
# so we do this manually for now in the NEXT 3 cells below
# from proveit import x
# from proveit.logic import Exists
# Exists(x, InSet(x, A_zero)).conclude_via_example(selected_tuple)

In [ ]:
from proveit import x
from proveit.logic import Exists
something_exists_in_A_zero = Exists(x, InSet(x, A_zero))

In [ ]:
from proveit.logic.booleans.quantification.existence import existence_by_example_single
existence_by_example_single

In [ ]:
from proveit import P, Lambda
_P_sub = Lambda(x, something_exists_in_A_zero.instance_expr)
_x_sub = selected_tuple # we've now proven this is a valid example element
existence_by_example_single_inst = existence_by_example_single.instantiate(
            {P:_P_sub, x:_x_sub})

##### And thus, $A_{zero} = \{(0, \ell, 0) | \ell \in \{0, \ldots, n-1\}\}$ is non-empty:

In [ ]:
from proveit.logic import NotEquals
from proveit.logic.sets import EmptySet
NotEquals(A_zero, EmptySet).prove()

##### Next, we extend the non-emptiness conclusion to the union.

In [ ]:
from proveit import m
# notice here we MIGHT actually need the domain to be Interval(1-delta_{}-delta_{}, 0)
B_zero = SetOfAll(m, ExprTuple(m, zero, zero), domain=Interval(one, zero))

In [ ]:
from proveit.logic.sets import Union
NotEquals(Union(A_zero, B_zero), EmptySet).prove()

##### And then we extend the non-emptiness claim to the `UnionAll`

In [ ]:
# temporarily trying this with the simplification, which then un-nests the subtracted deltas
from proveit import t
temp_exists_claim = Exists(t, NotEquals(
    Union(RoughBoundaryBVertices.definition().rhs.instance_expr.operands[0],
          RoughBoundaryBVertices.definition().rhs.instance_expr.operands[1]), EmptySet),
    domain=Interval(zero, _n)).simplification().rhs

##### We need some extra information to evaluate those Kronecker deltas:

In [ ]:
NotEquals(_n, zero).prove()

In [ ]:
NotEquals(subtract(_n, one), zero).prove()

In [ ]:
# this doesn't seem to help enough
# from proveit.logic import Equals
# Equals(temp_exists_claim.instance_expr.lhs, Union(A_zero, B_zero)).prove(assumptions=[Equals(t, zero)])

##### Apparently when then trying to call `temp_exists_claim.conclude_via_example(zero)`, the system is having trouble automatically connecting the original `SetOfAll`s with the `SetOfAll`s with $t= 0$, possibly because of some processing failures for the Kronecker deltas. So we try to manually provide/prompt an equation connecting the original union of `SsetOfAll`s with the version when $t=0$. (It's possible that this could be caught earlier, for example in cell 19 above, when formulating the zero-version of the `SetOfAll`s.)

In [ ]:
from proveit import X
from proveit.numbers import KroneckerDelta
unsimplified_expr = Union(SetOfAll(ell, ExprTuple(zero, ell, zero), domain=Interval(subtract(one, KroneckerDelta(zero, zero)), subtract(subtract(_n, one), zero))),
                          SetOfAll(m, ExprTuple(m, zero, zero), domain=Interval(subtract(subtract(one, KroneckerDelta(zero, subtract(_n, one))), KroneckerDelta(zero, _n)), zero)))

In [ ]:
from proveit.logic import Equals
current_eq = Equals(unsimplified_expr, unsimplified_expr).prove()

In [ ]:
(current_eq.inner_expr().rhs.operands[0].domain.lower_bound.operands[1].operand.simplify().
    inner_expr().rhs.operands[0].domain.lower_bound.simplify().
    inner_expr().rhs.operands[0].domain.upper_bound.simplify().
    inner_expr().rhs.operands[1].domain.lower_bound.operands[0].operands[1].operand.simplify().
    inner_expr().rhs.operands[1].domain.lower_bound.operands[1].operand.simplify().
    inner_expr().rhs.operands[1].domain.lower_bound.simplify()
)

In [ ]:
# this might have to be run TWICE. Sadly.
temp_exists_claim.conclude_via_example(zero)

In [ ]:
NotEquals(union_all, EmptySet).prove()

In [ ]:
%qed